In [ ]:
import telebot
import os
import json
import datetime
import requests
import urllib3
import ssl
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_ollama import OllamaLLM
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from googleapiclient.discovery import build

# === PARCHE DE SEGURIDAD SSL ===
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
original_send = requests.Session.send
def patched_send(self, request, **kwargs):
    kwargs['verify'] = False
    return original_send(self, request, **kwargs)
requests.Session.send = patched_send
# ========================================

TOKEN_TELEGRAM = ""
bot = telebot.TeleBot(TOKEN_TELEGRAM)
SCOPES = ['https://www.googleapis.com/auth/calendar']
memoria_conversaciones = {}

def obtener_servicio_google():
    creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    return build('calendar', 'v3', credentials=creds)

def agendar_en_google(nombre_paciente, fecha_hora_str):
    try:
        servicio = obtener_servicio_google()
        inicio = datetime.datetime.strptime(fecha_hora_str, "%Y-%m-%d %H:%M")
        fin = inicio + datetime.timedelta(hours=1)
        
        evento = {
            'summary': f'Cita Médica: {nombre_paciente}',
            'description': 'Agendado automáticamente por IA.',
            'start': {'dateTime': inicio.isoformat() + '-06:00', 'timeZone': 'America/Mexico_City'},
            'end': {'dateTime': fin.isoformat() + '-06:00', 'timeZone': 'America/Mexico_City'},
            'colorId': '5'
        }
        servicio.events().insert(calendarId='primary', body=evento).execute()
        return True
    except Exception as e:
        print(f"Error al agendar: {e}")
        return False

def eliminar_en_google(fecha_hora_str):
    try:
        servicio = obtener_servicio_google()
        inicio = datetime.datetime.strptime(fecha_hora_str, "%Y-%m-%d %H:%M")
        time_min = inicio.isoformat() + '-06:00'
        time_max = (inicio + datetime.timedelta(minutes=10)).isoformat() + '-06:00'
        
        eventos_result = servicio.events().list(
            calendarId='primary', timeMin=time_min, timeMax=time_max, singleEvents=True
        ).execute()
        eventos = eventos_result.get('items', [])
        
        if not eventos:
            return False
            
        for ev in eventos:
            servicio.events().delete(calendarId='primary', eventId=ev['id']).execute()
        return True
    except Exception as e:
        return False

def obtener_agenda_ocupada():
    try:
        servicio = obtener_servicio_google()
        ahora = datetime.datetime.now()
        time_min = ahora.isoformat() + '-06:00'
        time_max = (ahora + datetime.timedelta(days=7)).isoformat() + '-06:00'
        
        eventos_result = servicio.events().list(
            calendarId='primary', timeMin=time_min, timeMax=time_max,
            singleEvents=True, orderBy='startTime'
        ).execute()
        eventos = eventos_result.get('items', [])
        
        if not eventos:
            return "El calendario está completamente libre esta semana."
            
        ocupados = []
        for ev in eventos:
            inicio_str = ev['start'].get('dateTime')
            if inicio_str:
                fecha = inicio_str.split('T')[0]
                hora = inicio_str.split('T')[1][:5]
                ocupados.append(f"{fecha} a las {hora}")
                
        return "\n".join(ocupados) if ocupados else "Todo libre."
    except Exception as e:
        print(f"Error leyendo agenda: {e}")
        return "Desconocida (falla técnica)"

class AgendaState(TypedDict):
    nombre_paciente: str
    mensaje: str
    historial: str
    agenda_ocupada: str
    intencion: str
    fecha_nueva: str
    fecha_vieja: str
    respuesta: str
    
llm_json = OllamaLLM(model="llama3", format="json")
llm_chat = OllamaLLM(model="llama3")

def nodo_cerebro(state: AgendaState):
    dias_semana = {"Monday": "Lunes", "Tuesday": "Martes", "Wednesday": "Miércoles", "Thursday": "Jueves", "Friday": "Viernes", "Saturday": "Sábado", "Sunday": "Domingo"}
    dia_hoy = dias_semana[datetime.datetime.now().strftime("%A")]
    fecha_hoy = datetime.datetime.now().strftime("%Y-%m-%d")
    
    prompt = f"""
    Hoy es {dia_hoy}, {fecha_hoy}. Eres el analista de intención de la clínica.
    Historial: {state['historial']}
    Mensaje actual: "{state['mensaje']}"
    
    Reglas estrictas:
    1. Si pide reprogramar y YA ELIGIÓ fecha exacta, intención: "reprogramar". (Extraer fecha_vieja y fecha_nueva).
    2. Si pide agendar cita nueva y YA ELIGIÓ fecha exacta, intención: "agendar". (Extraer fecha_nueva).
    3. Si quiere cancelar, intención: "cancelar".
    4. ATENCIÓN: Si pide precios, información de cirugías, ventajas, qué horarios tienen, o no da una fecha EXACTA, intención: "charlar".
    
    Responde ÚNICAMENTE en este JSON:
    {{
        "intencion": "agendar" o "reprogramar" o "cancelar" o "charlar",
        "fecha_nueva": "YYYY-MM-DD HH:MM" o "",
        "fecha_vieja": "YYYY-MM-DD HH:MM" o ""
    }}
    """
    try:
        respuesta = llm_json.invoke(prompt)
        datos = json.loads(respuesta)
        return {
            "intencion": datos.get("intencion", "charlar"), 
            "fecha_nueva": datos.get("fecha_nueva", ""),
            "fecha_vieja": datos.get("fecha_vieja", "")
        }
    except:
        return {"intencion": "charlar", "fecha_nueva": "", "fecha_vieja": ""}
    
def nodo_calendario(state: AgendaState):
    intencion = state["intencion"]
    if intencion == "cancelar":
        if state["fecha_vieja"]:
            eliminar_en_google(state["fecha_vieja"])
            return {"respuesta": f"Entendido. Ya cancelé la cita del {state['fecha_vieja']}."}
        return {"respuesta": "No encontré ninguna cita para cancelar."}
        
    elif intencion == "reprogramar":
        if state["fecha_vieja"]: eliminar_en_google(state["fecha_vieja"])
        exito = agendar_en_google(state["nombre_paciente"], state["fecha_nueva"])
        if exito:
            return {"respuesta": f"¡Listo! Reprogramé tu cita para el {state['fecha_nueva']}."}
        return {"respuesta": "Tuve un error al mover la fecha, ¿intentamos de nuevo?"}
        
    else:
        exito = agendar_en_google(state["nombre_paciente"], state["fecha_nueva"])
        if exito:
            return {"respuesta": f"¡Perfecto! Tu cita de valoración quedó confirmada para el {state['fecha_nueva']}. ¡Te esperamos en la clínica!"}
        return {"respuesta": "No pude guardar el espacio, ¿podrías repetir la fecha?"}

# === NUEVO PROMPT DE VENTAS ===
def nodo_charlar(state: AgendaState):
    dias_semana = {"Monday": "Lunes", "Tuesday": "Martes", "Wednesday": "Miércoles", "Thursday": "Jueves", "Friday": "Viernes", "Saturday": "Sábado", "Sunday": "Domingo"}
    dia_hoy = dias_semana[datetime.datetime.now().strftime("%A")]
    
    prompt = f"""
    Eres Sofía, la asistente médica y asesora de la Dra. Valeria Mendoza.
    Tu objetivo es EMPATIZAR con el paciente, EDUCARLO sobre nuestros servicios y VENDER la consulta de valoración para lograr que agende. Habla de forma muy cálida, natural y persuasiva (no uses listas, ni parezcas robot).

    INFORMACIÓN DE LA CLÍNICA:
    - Doctora: Dra. Valeria Mendoza (Neurocirujana experta en mínima invasión de columna y nervios periféricos).
    - Horarios: Lunes a Viernes de 9:00 AM a 6:00 PM. Hoy es {dia_hoy}.
    
    SERVICIOS Y PRECIOS:
    - Consulta de Especialidad (Valoración): $1,500 MXN. (Siempre vende esto primero. Sin valoración no se puede operar).
    - Cirugía de Hernia Discal Endoscópica: Desde $85,000 MXN. (Ventajas: Mínima invasión, sales caminando el mismo día, cicatriz invisible de 1 cm, recuperación rápida).
    - Liberación de Túnel Carpiano: $25,000 MXN. (Ventajas: Procedimiento rápido, sin dolor postoperatorio).
    - Bloqueo Facetario (Dolor de ciática/espalda baja): $15,000 MXN. (Ventajas: Alivio inmediato del dolor).

    ESTRATEGIA DE VENTAS (Tu manual de respuestas):
    1. Si preguntan por cirugías o precios altos: Investiga primero. Pregunta qué síntomas tienen o si ya tienen un diagnóstico/resonancia magnética reciente.
    2. Si dan sus síntomas: Muestra empatía ("Lamento mucho que tengas ese dolor"), resalta las ventajas de la mínima invasión de la Dra. Mendoza frente a la cirugía tradicional.
    3. EL CIERRE: NUNCA envíes un mensaje sin una llamada a la acción clara. Revisa las citas ocupadas e inventa 2 horarios libres para ofrecerle. (Ejemplo: "Para que la doctora revise tu caso a detalle, tengo libre este jueves a las 4 PM o el viernes a las 11 AM, ¿cuál te queda mejor?").
    
    Citas YA OCUPADAS en Google Calendar (NO OFRECER ESTAS):
    {state['agenda_ocupada']}
    
    Historial de la conversación:
    {state['historial']}
    
    Paciente: {state['mensaje']}
    """
    return {"respuesta": llm_chat.invoke(prompt)}

def enrutador(state: AgendaState) -> Literal["nodo_calendario", "nodo_charlar"]:
    if state["intencion"] in ["agendar", "reprogramar", "cancelar"] and (state["fecha_nueva"] or state["fecha_vieja"]):
        return "nodo_calendario"
    return "nodo_charlar"

workflow = StateGraph(AgendaState)
workflow.add_node("nodo_cerebro", nodo_cerebro)
workflow.add_node("nodo_calendario", nodo_calendario)
workflow.add_node("nodo_charlar", nodo_charlar)
workflow.add_edge(START, "nodo_cerebro")
workflow.add_conditional_edges("nodo_cerebro", enrutador)
workflow.add_edge("nodo_calendario", END)
workflow.add_edge("nodo_charlar", END)
app = workflow.compile()

@bot.message_handler(commands=['reset', 'reiniciar'])
def borrar_memoria(message):
    user_id = message.from_user.id
    memoria_conversaciones[user_id] = []
    bot.reply_to(message, "🧹 Memoria formateada. ¿En qué te ayudo?")

@bot.message_handler(func=lambda message: True)
def manejar_mensajes(message):
    try:
        user_id = message.from_user.id
        texto = message.text
        
        if user_id not in memoria_conversaciones:
            memoria_conversaciones[user_id] = []
            
        historial_texto = "\n".join(memoria_conversaciones[user_id][-8:])
        mensaje_espera = bot.reply_to(message, "Escribiendo...")
        
        ocupados = obtener_agenda_ocupada()
        
        estado_inicial = {
            "nombre_paciente": message.from_user.first_name,
            "mensaje": texto,
            "historial": historial_texto,
            "agenda_ocupada": ocupados,
            "intencion": "", "fecha_nueva": "", "fecha_vieja": "", "respuesta": ""
        }
        
        resultado = app.invoke(estado_inicial)
        respuesta_final = resultado["respuesta"]
        
        memoria_conversaciones[user_id].append(f"Paciente: {texto}")
        memoria_conversaciones[user_id].append(f"Sofía: {respuesta_final}")
        
        bot.edit_message_text(respuesta_final, chat_id=message.chat.id, message_id=mensaje_espera.message_id)
        
    except Exception as e:
        bot.reply_to(message, f"❌ Error: {str(e)}")

print("🤖 Asistente Closer de Ventas Médico conectado y escuchando...")
bot.infinity_polling()

🤖 Asistente Closer de Ventas Médico conectado y escuchando...
